In [27]:
import hashlib
import re
import zipfile
from pathlib import Path
import duckdb
import pandas as pd

RAW = Path("../data/raw").resolve()
zips = sorted(RAW.glob("data_*.zip"))
[z.name for z in zips]

['data_2013.zip',
 'data_Q1_2016.zip',
 'data_Q1_2023.zip',
 'data_Q2_2023.zip',
 'data_Q2_2026.zip',
 'data_Q3_2023.zip']

In [28]:
con = duckdb.connect()

In [29]:
DATE_FILE = re.compile(r"(\d{4}-\d{2}-\d{2})\.csv$")

def normalised_fingerprint(cols):
    """Hash after trimming, lowercasing and sorting names (order doesn't matter)."""
    cleaned = sorted(c.strip().lower() for c in cols)
    return hashlib.sha256(",".join(cleaned).encode()).hexdigest()[:10]

rows = []
for zpath in zips:
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            match = DATE_FILE.search(info.filename)
            if match is None or "__MACOSX" in info.filename:
                rows.append({"zip": zpath.name, "member": info.filename,
                             "kind": "skipped (not a daily CSV)"})
                continue
            with zf.open(info) as f:
                header = f.readline().decode("utf-8-sig").strip()
            cols = header.split(",")
            rows.append({
                "zip": zpath.name,
                "member": info.filename,
                "kind": "daily csv",
                "date": match.group(1),
                "size_bytes": info.file_size,
                "n_cols": len(cols),
                "fingerprint": hashlib.sha256(header.encode()).hexdigest()[:10],
                "set_fingerprint": normalised_fingerprint(cols),
                "columns": cols,
            })

files = pd.DataFrame(rows)
files["kind"].value_counts()

kind
daily csv                    721
skipped (not a daily CSV)    639
Name: count, dtype: int64

In [30]:
files[files["kind"] != "daily csv"][["zip", "member"]]

,zip,member
0,data_2013.zip,__MACOSX/._2013
2,data_2013.zip,__MACOSX/2013/._2013-12-11.csv
4,data_2013.zip,__MACOSX/2013/._2013-12-05.csv
6,data_2013.zip,__MACOSX/2013/._2013-05-21.csv
8,data_2013.zip,__MACOSX/2013/._2013-05-09.csv
...,...,...
1351,data_Q3_2023.zip,__MACOSX/data_Q3_2023/._2023-07-24.csv
1353,data_Q3_2023.zip,__MACOSX/data_Q3_2023/._2023-07-26.csv
1355,data_Q3_2023.zip,__MACOSX/data_Q3_2023/._2023-09-23.csv
1357,data_Q3_2023.zip,__MACOSX/data_Q3_2023/._2023-09-22.csv


In [31]:
daily = files[files["kind"] == "daily csv"].sort_values("date").copy()
daily["n_cols"] = daily["n_cols"].astype(int)

print("eras by exact header:     ", daily["fingerprint"].nunique())
print("eras by normalised header:", daily["set_fingerprint"].nunique(), "\n")

eras = (
    daily.groupby("fingerprint")
    .agg(first_date=("date", "min"), last_date=("date", "max"),
         days=("date", "count"), n_cols=("n_cols", "first"))
    .sort_values("first_date")
)
eras

eras by exact header:      6
eras by normalised header: 6 



,first_date,last_date,days,n_cols
fingerprint,,,,
dac22a3754,2013-04-10,2013-12-31,266,85
9cba7b72d7,2016-01-01,2016-03-31,91,95
348825372a,2023-01-01,2023-03-31,90,179
f7b9eeba76,2023-04-01,2023-06-30,91,186
8024e7ddc2,2023-07-01,2023-09-30,92,193
2d9774ddbb,2026-04-01,2026-06-30,91,197


In [32]:
era_columns = daily.groupby("fingerprint")["columns"].first()

previous = None
for fp, era in eras.iterrows():
    cols = era_columns[fp]
    if previous is None:
        print(f"{era.first_date} → {era.last_date}: first era, {era.n_cols} columns\n")
    else:
        added   = [c for c in cols if c not in previous]
        removed = [c for c in previous if c not in cols]
        moved   = [c for c in cols if c in previous and cols.index(c) != previous.index(c)]
        print(f"{era.first_date} → {era.last_date} ({era.n_cols} columns)")
        print(f"   added:   {added}")
        print(f"   removed: {removed}")
        print(f"   columns that changed position: {len(moved)}\n")
    previous = cols

2013-04-10 → 2013-12-31: first era, 85 columns

2016-01-01 → 2016-03-31 (95 columns)
   added:   ['smart_22_normalized', 'smart_22_raw', 'smart_220_normalized', 'smart_220_raw', 'smart_222_normalized', 'smart_222_raw', 'smart_224_normalized', 'smart_224_raw', 'smart_226_normalized', 'smart_226_raw']
   removed: []
   columns that changed position: 54

2023-01-01 → 2023-03-31 (179 columns)
   added:   ['smart_16_normalized', 'smart_16_raw', 'smart_17_normalized', 'smart_17_raw', 'smart_18_normalized', 'smart_18_raw', 'smart_23_normalized', 'smart_23_raw', 'smart_24_normalized', 'smart_24_raw', 'smart_160_normalized', 'smart_160_raw', 'smart_161_normalized', 'smart_161_raw', 'smart_163_normalized', 'smart_163_raw', 'smart_164_normalized', 'smart_164_raw', 'smart_165_normalized', 'smart_165_raw', 'smart_166_normalized', 'smart_166_raw', 'smart_167_normalized', 'smart_167_raw', 'smart_168_normalized', 'smart_168_raw', 'smart_169_normalized', 'smart_169_raw', 'smart_170_normalized', 'smart_

In [33]:
daily[daily["size_bytes"] < 10_000][["zip", "date", "size_bytes"]]

,zip,date,size_bytes


In [34]:
daily["d"] = pd.to_datetime(daily["date"])
for zip_name, grp in daily.groupby("zip"):
    all_days = pd.date_range(grp["d"].min(), grp["d"].max())
    missing = sorted(set(all_days) - set(grp["d"]))
    print(f"{zip_name}: {len(grp)} days, missing: {[m.date() for m in missing]}")

data_2013.zip: 266 days, missing: []
data_Q1_2016.zip: 91 days, missing: []
data_Q1_2023.zip: 90 days, missing: []
data_Q2_2023.zip: 91 days, missing: []
data_Q2_2026.zip: 91 days, missing: []
data_Q3_2023.zip: 92 days, missing: []


In [35]:
def expected_days(zip_name):
    m = re.match(r"data_Q(\d)_(\d{4})\.zip", zip_name)
    if m:
        q, year = int(m.group(1)), int(m.group(2))
        start = pd.Timestamp(year=year, month=3 * q - 2, day=1)
        return pd.date_range(start, start + pd.offsets.QuarterEnd(0))
    year = int(re.match(r"data_(\d{4})\.zip", zip_name).group(1))
    return pd.date_range(f"{year}-01-01", f"{year}-12-31")

for zip_name, grp in daily.groupby("zip"):
    missing = sorted(set(expected_days(zip_name)) - set(grp["d"]))
    print(f"{zip_name}: missing {len(missing)} days; first few: {[m.date() for m in missing[:5]]}")

data_2013.zip: missing 99 days; first few: [datetime.date(2013, 1, 1), datetime.date(2013, 1, 2), datetime.date(2013, 1, 3), datetime.date(2013, 1, 4), datetime.date(2013, 1, 5)]
data_Q1_2016.zip: missing 0 days; first few: []
data_Q1_2023.zip: missing 0 days; first few: []
data_Q2_2023.zip: missing 0 days; first few: []
data_Q2_2026.zip: missing 0 days; first few: []
data_Q3_2023.zip: missing 0 days; first few: []


In [36]:
import csv

HIST = Path("../contracts/reference/Drive_Stats_Schema_2018_Onward.csv").resolve()

doc_by_quarter = {}
with open(HIST, encoding="utf-8-sig") as f:
    for row in csv.reader(f):
        if row and row[0].strip():
            doc_by_quarter[row[0].strip()] = [c.strip() for c in row[1:] if c.strip()]

def quarter_label(zip_name):
    m = re.match(r"data_Q(\d)_(\d{4})\.zip", zip_name)
    return f"{m.group(2)} Q{m.group(1)}" if m else None

for zip_name, grp in daily.groupby("zip"):
    expected = doc_by_quarter.get(quarter_label(zip_name))
    if expected is None:
        print(f"{zip_name}: not covered by the history file")
        continue
    differ = grp[grp["columns"].apply(lambda cols: cols != expected)]
    print(f"{zip_name}: {len(grp)} days, {len(differ)} days differ from the doc")

data_2013.zip: not covered by the history file
data_Q1_2016.zip: not covered by the history file
data_Q1_2023.zip: 90 days, 0 days differ from the doc
data_Q2_2023.zip: 91 days, 0 days differ from the doc
data_Q2_2026.zip: not covered by the history file
data_Q3_2023.zip: 92 days, 0 days differ from the doc


In [37]:
REF = Path("../contracts/reference").resolve()
current_doc = (REF / "Drive_Stats_Schema_Current.csv").read_text(encoding="utf-8-sig").splitlines()
doc_cols = [line.strip() for line in current_doc[1:] if line.strip()]

latest_cols = daily.iloc[-1]["columns"]

print("doc columns:", len(doc_cols), "| real columns (latest day):", len(latest_cols))
print("in doc but not in data:", [c for c in doc_cols if c not in latest_cols])
print("in data but not in doc:", [c for c in latest_cols if c not in doc_cols])
print("same order:", doc_cols == latest_cols)


doc columns: 197 | real columns (latest day): 197
in doc but not in data: ['smart_211_normailized', 'smart_212_normailized']
in data but not in doc: ['smart_211_normalized', 'smart_212_normalized']
same order: False


In [38]:
def profile_codes(day, cols):
    path = Path(f"../data/sample/{day}.csv").resolve()
    rel = f"read_csv('{path}', all_varchar=true)"
    present = con.sql(f"SELECT * FROM {rel} LIMIT 0").columns
    for c in cols:
        if c not in present:
            print(f"{day}: {c} does not exist yet\n")
            continue
        print(f"{day}: {c}")
        print(con.sql(f"""
            SELECT
                count(*)  AS rows,
                count({c}) AS filled,
                sum(CASE WHEN {c} IS NOT NULL AND TRY_CAST({c} AS BIGINT) IS NULL THEN 1 ELSE 0 END) AS not_a_number,
                sum(CASE WHEN {c} LIKE '0%' AND length({c}) > 1 THEN 1 ELSE 0 END) AS leading_zero,
                min({c}) AS smallest_text,
                max({c}) AS largest_text
            FROM {rel}
        """).df(), "\n")

codes = ["vault_id", "pod_id", "pod_slot_num", "cluster_id", "datacenter", "is_legacy_format"]
for day in ["2023-04-01", "2023-07-01", "2026-04-01"]:
    profile_codes(day, codes)

2023-04-01: vault_id
     rows  filled  not_a_number  leading_zero smallest_text largest_text
0  241681  241681           0.0           0.0          1000         3003 

2023-04-01: pod_id
     rows  filled  not_a_number  leading_zero smallest_text largest_text
0  241681  241681           0.0      120873.0            00           19 

2023-04-01: pod_slot_num does not exist yet

2023-04-01: cluster_id does not exist yet

2023-04-01: datacenter does not exist yet

2023-04-01: is_legacy_format
     rows  filled  not_a_number  leading_zero smallest_text largest_text
0  241681  241681      241681.0           0.0         False        False 

2023-07-01: vault_id
     rows  filled  not_a_number  leading_zero smallest_text largest_text
0  245757  245757           0.0           0.0          1000         3003 

2023-07-01: pod_id
     rows  filled  not_a_number  leading_zero smallest_text largest_text
0  245757  245757           0.0      122878.0            00           19 

2023-07-01: pod_slot

In [39]:
d26 = Path("../data/sample/2026-04-01.csv").resolve()
d23 = Path("../data/sample/2023-07-01.csv").resolve()

print("2026: models of drives with NO pod_slot_num")
print(con.sql(f"""
    SELECT model, count(*) AS drives
    FROM read_csv('{d26}', all_varchar=true)
    WHERE pod_slot_num IS NULL
    GROUP BY ALL ORDER BY drives DESC LIMIT 10
""").df(), "\n")

print("2023-07-01: which vaults have a blank datacenter?")
print(con.sql(f"""
    SELECT vault_id,
           count(*) AS drives,
           count(datacenter) AS with_datacenter
    FROM read_csv('{d23}', all_varchar=true)
    GROUP BY ALL
    HAVING count(*) > count(datacenter)
    ORDER BY drives DESC
""").df())

2026: models of drives with NO pod_slot_num
                                    model  drives
0  Seagate BarraCuda 120 SSD ZA250CM10003    1070
1                          CT250MX500SSD1     698
2      Seagate BarraCuda SSD ZA250CM10002     472
3                             DELLBOSS VD     431
4           Seagate IronWolf ZA250NM10002     189
5                         WDC WDS250G2B0A     170
6                              ST500LM030     155
7                 WD Blue SA510 2.5 250GB     144
8                      TOSHIBA MQ01ABF050     110
9                             Seagate SSD      98 

2023-07-01: which vaults have a blank datacenter?
  vault_id  drives  with_datacenter
0     1176    1220                0
1     1167    1220                0
2     1402    1220                0
3     1172    1220                0
4     1403    1220                0
